<a href="https://colab.research.google.com/github/Ruturaj2472/AI-Support-Ticket-Triage-System/blob/main/AI_Support_Ticket_Triage_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Support Ticket Triage System

Businesses receive customer support tickets from email, chat, WhatsApp, forms, or helpdesk tools.
Each ticket must be **understood, categorized, prioritized, routed to the correct team, and replied to professionally.**
Doing this manually is slow and inconsistent.

This notebook builds a simple **AI-powered support ticket triage system** that, for every incoming ticket:

1. Accepts a customer support message
2. Classifies the ticket **category**
3. Detects the **priority** level
4. Assigns the correct **department**
5. Generates a short **issue summary**
6. Generates a **customer-facing reply**
7. Decides whether **human escalation** is required

The system is built using an **intent-based routing workflow**: the LLM first classifies the intent/category of the
ticket (this is the "routing" step), and that classification drives every downstream decision (priority, department,
reply tone, escalation).

We support **three interchangeable LLM providers** — OpenAI, Groq (free), and Gemini (free) — through **one common
function** so you can switch providers with a single parameter, without changing any other code.

We also maintain a **conversation memory dictionary** so that if a customer sends follow-up messages on the same
ticket, the model has the full context of the conversation so far.


## 1. Install and Setup

In [1]:
# Install the official OpenAI Python SDK
# Used to call OpenAI models (and also compatible with Groq's OpenAI-style endpoint if needed)
!pip install openai -q

# Install the official Groq Python SDK
# Groq offers a generous FREE tier with very fast inference (Llama, Gemma, etc. models)
!pip install groq -q

# Install the official Google Generative AI SDK
# Used to call Gemini models, which also have a free tier
!pip install google-generativeai -q

# Install Gradio to build a simple chatbot-style web UI at the end of the notebook
!pip install gradio -q

# Pydantic is used to define strict, structured output schemas for the LLM responses
# (Usually pre-installed in Colab, but installing here to be safe)
!pip install pydantic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


## 2. Imports

In [2]:
# Standard library import used to read environment variables if needed
import os

# json is used to parse/validate structured JSON responses coming back from
# providers that do not have native Pydantic parsing support (Groq, Gemini)
import json

# typing helpers
# Literal restricts a field to a fixed set of allowed values (used for category/priority/etc.)
# List / Dict are used for type-hinting conversation history structures
# Optional marks a field that may or may not be present
from typing import Literal, List, Dict, Optional

# Pydantic is used to define a strict schema for what the model must return
# BaseModel = base class for structured data models
# Field = adds descriptions/validation/defaults to individual fields
from pydantic import BaseModel, Field

# OpenAI SDK client
from openai import OpenAI

# Groq SDK client (Groq's client library mirrors the OpenAI client design)
from groq import Groq

# Google Generative AI SDK (for Gemini models)
import google.generativeai as genai

# Used to securely read API keys stored in Colab Secrets
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 3. Initialize Clients for All Three Providers



In [3]:
# ---------------- OpenAI ----------------
# Model used for all OpenAI calls (small, cheap, good at structured output)
MODEL_OPENAI = "gpt-4.1-mini"

try:
    # Fetch the OpenAI key from Colab Secrets and create the client
    client_openai = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
except Exception as e:
    # If the secret isn't set, disable this provider gracefully
    print("OpenAI client not initialized:", e)
    client_openai = None


# ---------------- Groq ----------------
# Free, very fast open-weight model hosted by Groq
MODEL_GROQ = "llama-3.3-70b-versatile"

try:
    client_groq = Groq(api_key=userdata.get("GROQ_API_KEY"))
except Exception as e:
    print("Groq client not initialized:", e)
    client_groq = None


# ---------------- Gemini ----------------
# Free-tier Gemini model
MODEL_GEMINI = "gemini-2.5-flash"

try:
    # Gemini uses a global configure() call instead of a client object
    genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
    client_gemini_ready = True
except Exception as e:
    print("Gemini client not initialized:", e)
    client_gemini_ready = False

Groq client not initialized: Secret GROQ_API_KEY does not exist.
Gemini client not initialized: Secret GEMINI_API_KEY does not exist.


## 4. Pydantic Structured Output Schema

This is the single source of truth for what the model must output for every ticket. All three providers will be
asked to fill in exactly this schema, which keeps the rest of the code provider-agnostic.


In [4]:
# Structured output format for the Ticket Triage system
# Every provider (OpenAI / Groq / Gemini) will return data matching this exact schema
class TicketTriageOutput(BaseModel):

    # The ticket category, i.e. the "intent" of the customer message
    # This is the core of the intent-based routing workflow: every other
    # field (department, priority defaults, reply tone) is driven off this
    category: Literal[
        "billing_and_payments",   # Invoices, charges, refunds, payment failures
        "technical_issue",        # Bugs, errors, app/website not working
        "account_access",         # Login issues, password reset, locked account
        "product_inquiry",        # Questions about features, plans, how something works
        "complaint",              # Customer is unhappy about service/product experience
        "feature_request",        # Customer is suggesting a new feature/improvement
        "general_query",          # Anything else that doesn't fit the above
    ] = Field(description="The primary category/intent of the customer's ticket.")

    # Priority level of the ticket, used for queue ordering / SLA
    priority: Literal["low", "medium", "high", "urgent"] = Field(
        description="Urgency of the ticket. 'urgent' = active outage / data loss / security issue / payment charged incorrectly at scale."
    )

    # Department the ticket should be routed to
    department: Literal[
        "billing_team",
        "technical_support",
        "account_management",
        "sales_and_product",
        "customer_success",
        "general_support",
    ] = Field(description="The team that should own and resolve this ticket.")

    # A short, internal-facing summary of the issue (for agents, not the customer)
    issue_summary: str = Field(
        description="A concise 1-2 sentence internal summary of what the customer needs, for the support agent to scan quickly."
    )

    # The actual reply that will be shown to the customer
    customer_reply: str = Field(
        description="A short, professional, empathetic reply to send directly to the customer."
    )

    # Whether this ticket needs to be handed off to a human agent instead of being auto-resolved
    escalate_to_human: bool = Field(
        description="True if a human agent must personally handle this ticket (e.g. refunds, complaints, low-confidence classification, urgent priority)."
    )

    # Model's confidence in its own classification, used as a safety net in code
    confidence: float = Field(
        ge=0, le=1,
        description="Model's confidence (0 to 1) in the category classification above."
    )

## 5. System Prompt (Business Rules)

The system prompt encodes the triage policy: how to classify, how to decide priority/department, and — critically —
the exact rules for when a ticket must be escalated to a human. Every provider is given the same system prompt so
behavior stays consistent regardless of which model answers.


In [5]:
SYSTEM_PROMPT = """
You are an AI Support Ticket Triage Assistant for a software company.

For every customer message, you must:
1. Classify the ticket into exactly one of the allowed categories.
2. Assign a priority level based on urgency and business impact.
3. Assign the correct department to handle the ticket.
4. Write a short, internal issue summary (for the support agent, not the customer).
5. Write a short, professional, empathetic customer-facing reply.
6. Decide whether the ticket must be escalated to a human agent.
7. Provide a confidence score (0 to 1) for your category classification.

Allowed categories:
- billing_and_payments
- technical_issue
- account_access
- product_inquiry
- complaint
- feature_request
- general_query

Priority rules:
- urgent: active outage, data loss, security breach, or being charged incorrectly at scale
- high: customer cannot use the product/service at all (e.g. cannot log in, payment failed and blocked access)
- medium: partial issue, workaround exists, or a billing question with no active harm
- low: general question, feature request, or feedback with no urgency

Department mapping:
- billing_and_payments -> billing_team
- technical_issue -> technical_support
- account_access -> technical_support
- product_inquiry -> sales_and_product
- feature_request -> sales_and_product
- complaint -> customer_success
- general_query -> general_support

Escalation rules (escalate_to_human must be true if ANY of these apply):
- category is "complaint"
- priority is "urgent" or "high"
- the customer explicitly asks to speak to a human/agent/manager
- you are not confident about the classification (confidence below 0.6)
- the message mentions refunds, legal action, or threats to cancel/churn

Tone rules for customer_reply:
- Be warm, professional, and concise (2-4 sentences).
- Acknowledge the issue before providing next steps.
- Never promise a specific refund amount or timeline you are not certain about.
- If escalate_to_human is true, tell the customer a human agent will follow up shortly.

Always respond using ONLY the structured fields defined by the output schema. Do not invent extra fields.
"""

## 6. Conversation Memory

Real tickets often involve back-and-forth (customer replies again, adds details, etc.). We keep a simple
**in-memory dictionary** keyed by `ticket_id`, so each ticket's conversation history is preserved across turns and
fed back into the model on every new message — this is what allows continuous, context-aware conversations.

The memory is stored in one **canonical format** (`role`: `"user"` or `"assistant"`, `content`: text) regardless of
provider. Each provider-specific function below converts this canonical history into whatever format that provider's
API expects.


In [6]:
# conversation_memory maps: ticket_id -> list of {"role": "user"/"assistant", "content": "..."}
# This dict lives in Python memory for the lifetime of the Colab session/runtime
conversation_memory: Dict[str, List[Dict[str, str]]] = {}


def get_history(ticket_id: str) -> List[Dict[str, str]]:
    """Return the stored conversation history for a ticket_id, creating an empty one if new."""
    if ticket_id not in conversation_memory:
        conversation_memory[ticket_id] = []
    return conversation_memory[ticket_id]


def update_history(ticket_id: str, user_message: str, assistant_reply: str) -> None:
    """Append the latest user message and assistant reply to a ticket's memory."""
    history = get_history(ticket_id)
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": assistant_reply})

## 7. Build Provider-Specific Input Messages

OpenAI and Groq both use the familiar `{"role": ..., "content": ...}` chat message list with a `"system"` role.
Gemini instead takes a separate `system_instruction` and expects history turns with roles `"user"` / `"model"`
(not `"assistant"`). This function converts our canonical memory into the exact shape each provider needs.


In [7]:
def build_input_messages(
    ticket_id: str,
    user_message: str,
    provider: Literal["openai", "groq", "gemini"],
):
    """
    Build the input payload for the given provider, combining:
    - the system prompt (business rules)
    - the ticket's prior conversation history (from conversation_memory)
    - the new incoming user message

    Returns a structure ready to be passed straight into that provider's API call.
    """

    # Pull whatever history already exists for this ticket (empty list if it's a new ticket)
    history = get_history(ticket_id)

    if provider in ("openai", "groq"):
        # Both OpenAI and Groq use OpenAI-style chat messages:
        # [{"role": "system"/"user"/"assistant", "content": "..."}]
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        # Replay prior turns exactly as stored (roles already match: user/assistant)
        for turn in history:
            messages.append({"role": turn["role"], "content": turn["content"]})

        # Add the new incoming customer message
        messages.append({"role": "user", "content": user_message})
        return messages

    elif provider == "gemini":
        # Gemini expects a list of {"role": "user"/"model", "parts": ["..."]}
        # Note: Gemini uses "model" instead of "assistant", and takes system
        # instructions separately (passed when creating the GenerativeModel)
        contents = []
        for turn in history:
            gemini_role = "model" if turn["role"] == "assistant" else "user"
            contents.append({"role": gemini_role, "parts": [turn["content"]]})

        # Add the new incoming customer message
        contents.append({"role": "user", "parts": [user_message]})
        return contents

    else:
        raise ValueError(f"Unknown provider: {provider}")

## 8. Provider-Specific Call Functions

Each function below sends the built messages to its respective provider and returns a validated
`TicketTriageOutput` object. Keeping these separate (instead of one giant if/else) makes each provider's quirks
easy to see and maintain independently.


In [8]:
def call_openai(messages) -> TicketTriageOutput:
    """Call OpenAI using the Responses API, which natively parses into our Pydantic schema."""

    response = client_openai.responses.parse(
        model=MODEL_OPENAI,
        input=messages,
        text_format=TicketTriageOutput,
    )

    # response.output_parsed is already a validated TicketTriageOutput instance
    return response.output_parsed

In [9]:
def call_groq(messages) -> TicketTriageOutput:
    """
    Call Groq's chat completion endpoint.
    Groq does not have native Pydantic parsing (like OpenAI's .responses.parse),
    so we request JSON mode and manually validate the returned JSON against our schema.
    """

    # Ask the model to output a JSON object matching our schema's field names.
    # We inject a short instruction + the schema so the model knows the exact shape to return.
    schema_hint = {
        "role": "system",
        "content": (
            "Respond ONLY with a single valid JSON object with exactly these keys: "
            "category, priority, department, issue_summary, customer_reply, "
            "escalate_to_human (boolean), confidence (float 0-1). "
            "Do not include markdown code fences or any extra text."
        ),
    }

    response = client_groq.chat.completions.create(
        model=MODEL_GROQ,
        messages=messages + [schema_hint],
        response_format={"type": "json_object"},
        temperature=0.2,
    )

    raw_text = response.choices[0].message.content

    # Validate/parse the raw JSON string into our strict Pydantic schema
    # This will raise a clear validation error if the model returns something malformed
    return TicketTriageOutput.model_validate_json(raw_text)

In [10]:
def call_gemini(contents) -> TicketTriageOutput:
    """
    Call Gemini using its native structured-output support:
    passing response_schema directly lets Gemini return JSON matching our Pydantic model.
    """

    model = genai.GenerativeModel(
        model_name=MODEL_GEMINI,
        system_instruction=SYSTEM_PROMPT,
    )

    response = model.generate_content(
        contents,
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=TicketTriageOutput,
        ),
    )

    # response.text contains the raw JSON string matching TicketTriageOutput
    return TicketTriageOutput.model_validate_json(response.text)

## 9. Unified Router Function (Intent-Based Routing Workflow)

This is the **one common function** the rest of the app (console loop, Gradio UI) calls. Given a `provider` name,
a `ticket_id`, and the new `user_message`, it:

1. Builds provider-specific input (Section 7)
2. Routes to the correct provider function (Section 8) — this "provider routing" mirrors the same intent-routing
   idea used inside the prompt (route based on a category), just one level up (route based on provider name)
3. Applies a **Python-side safety net**: if the model is unsure (`confidence < 0.6`) or misses an escalation
   rule, we force `escalate_to_human = True` regardless of what the model said
4. Updates the conversation memory for that ticket
5. Returns the final `TicketTriageOutput`


In [11]:
def get_triage_response(
    provider: Literal["openai", "groq", "gemini"],
    ticket_id: str,
    user_message: str,
) -> TicketTriageOutput:
    """
    Single entry point for the whole app: builds the provider-specific input,
    routes the call to the right provider function, applies a safety-net escalation
    check, updates memory, and returns the structured result.
    """

    # Step 1: build input in the exact shape this provider expects, including memory
    messages = build_input_messages(ticket_id, user_message, provider)

    # Step 2: route to the correct provider function
    if provider == "openai":
        if client_openai is None:
            raise RuntimeError("OpenAI client is not configured. Set OPENAI_API_KEY in Colab Secrets.")
        result = call_openai(messages)

    elif provider == "groq":
        if client_groq is None:
            raise RuntimeError("Groq client is not configured. Set GROQ_API_KEY in Colab Secrets.")
        result = call_groq(messages)

    elif provider == "gemini":
        if not client_gemini_ready:
            raise RuntimeError("Gemini client is not configured. Set GEMINI_API_KEY in Colab Secrets.")
        result = call_gemini(messages)

    else:
        raise ValueError(f"Unsupported provider: {provider}")

    # Step 3: extra safety layer in Python code (in case the model forgets a rule)
    # Force human escalation if confidence is low, even if the model said otherwise
    if result.confidence < 0.6:
        result.escalate_to_human = True

    # Force human escalation for complaints and urgent/high priority tickets too,
    # as a hard guarantee independent of model behavior
    if result.category == "complaint" or result.priority in ("urgent", "high"):
        result.escalate_to_human = True

    # Step 4: update the ticket's conversation memory with this turn
    update_history(ticket_id, user_message, result.customer_reply)

    # Step 5: return the final structured triage result
    return result

## 10. Main Loop (Console Testing)

A simple interactive loop to test the triage system in the notebook itself, before wiring up the Gradio UI.
Type `exit`, `quit`, or `bye` to stop.


In [12]:
def main():
    """Run an interactive console session for the Ticket Triage system."""

    print("AI Support Ticket Triage System")
    print("Type 'exit' to stop.\n")

    # Pick which provider to test with (must be configured with a valid API key above)
    provider = "openai"  # change to "groq" or "gemini" as needed

    # Use a single ticket_id for this console session so memory persists across turns
    ticket_id = "console-session-1"

    while True:
        user_message = input("Customer: ").strip()

        if user_message.lower() in ["exit", "quit", "bye"]:
            print("Session ended.")
            break

        if not user_message:
            print("Please type a message.")
            continue

        try:
            result = get_triage_response(provider, ticket_id, user_message)

            print(f"\nCategory        : {result.category}")
            print(f"Priority        : {result.priority}")
            print(f"Department      : {result.department}")
            print(f"Issue Summary   : {result.issue_summary}")
            print(f"Escalate?       : {result.escalate_to_human}")
            print(f"Confidence      : {result.confidence}")
            print(f"Customer Reply  : {result.customer_reply}\n")

        except Exception as e:
            print("Error while processing ticket:", e)

In [13]:
# Uncomment the line below to run the interactive console loop
main() # click the run button again to stop it from running

AI Support Ticket Triage System
Type 'exit' to stop.

Customer: Hi, I need some help.

Category        : general_query
Priority        : low
Department      : general_support
Issue Summary   : Customer has reached out for unspecified assistance.
Escalate?       : False
Confidence      : 0.8
Customer Reply  : Thank you for reaching out! Could you please provide more details about the help you need so we can assist you better?

Customer: I was charged twice for my subscription this month, can you check?

Category        : billing_and_payments
Priority        : medium
Department      : billing_team
Issue Summary   : Customer reports being charged twice for their subscription this month and requests verification.
Escalate?       : True
Confidence      : 0.95
Customer Reply  : Thank you for bringing this to our attention. We will review your subscription charges and get back to you shortly to resolve the issue. A human agent will follow up with you soon.

Customer: The app keeps crashing ev

In [14]:
# A handful of sample tickets covering every category, useful for quick manual testing
# or as example buttons in the Gradio UI further below
sample_tickets_for_testing = [
    "Hi, I need some help.",
    "I was charged twice for my subscription this month, can you check?",
    "The app keeps crashing every time I try to upload a file.",
    "I forgot my password and the reset email never arrives.",
    "What's the difference between your Pro and Enterprise plans?",
    "This is the third time I'm facing the same issue, I'm extremely frustrated.",
    "It would be great if you added dark mode to the dashboard.",
    "Can I get a refund for my last payment? This is unacceptable.",
    "I'd like to speak to a manager right now.",
    "Just checking, do you have a mobile app as well?",
]

## 11. Gradio UI

A lightweight chatbot-style web UI so non-technical users (or your teammates) can try the triage system without
touching code. It exposes a **provider dropdown** (to switch between OpenAI / Groq / Gemini live) and a
**ticket ID box** (so memory/history is tied to a specific ticket, just like a real helpdesk thread).


In [15]:
# Import Gradio, used to quickly build a web-based UI for the chatbot
import gradio as gr

In [16]:
def chatbot_interface(user_message, history, provider, ticket_id):
    """
    Function wired to the Gradio ChatInterface.
    Gradio manages the *displayed* chat bubbles itself; our own conversation_memory
    dict (keyed by ticket_id) is what actually gets sent back to the LLM for context,
    so switching the ticket_id box effectively starts a fresh conversation thread.
    """

    if not ticket_id:
        ticket_id = "default-ticket"

    try:
        result = get_triage_response(provider, ticket_id, user_message)
    except Exception as e:
        return f"⚠️ Error: {e}"

    # Build a readable response combining the internal triage details and the customer reply
    # (in a real deployment, the internal fields would go to an agent dashboard, and only
    # customer_reply would be shown to the customer -- shown together here for demo purposes)
    response_text = (
        f"**Category:** {result.category}  \n"
        f"**Priority:** {result.priority}  \n"
        f"**Department:** {result.department}  \n"
        f"**Escalate to human:** {result.escalate_to_human}  \n"
        f"**Internal summary:** {result.issue_summary}  \n\n"
        f"**Reply to customer:**  \n{result.customer_reply}"
    )
    return response_text

In [20]:
# Create the Gradio ChatInterface for the Ticket Triage system
demo = gr.ChatInterface(
    fn=chatbot_interface,

    title="AI Support Ticket Triage System",

    description="Type a customer message below. The system will classify, prioritize, route, "
                "summarize, and draft a reply -- powered by OpenAI, Groq, or Gemini (your choice).",

    # Extra inputs shown alongside the chat box: provider selector and ticket id
    additional_inputs=[
        gr.Dropdown(choices=["openai", "groq", "gemini"], value="openai", label="LLM Provider"),
        gr.Textbox(value="ticket-001", label="Ticket ID (used for conversation memory)"),
    ],

    examples=[[q] for q in sample_tickets_for_testing],

    cache_examples=False,
)

# Launch the Gradio web app
# In Colab this produces a local (and optionally public) link to open the chatbot UI
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc0fe8fabe021a7791.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# AI Support Ticket Triage System

This project implements an **AI-powered support ticket triage system** designed to automate and streamline the process of handling customer support inquiries. By leveraging large language models (LLMs), the system can intelligently process incoming support tickets, saving time and ensuring consistency.

## Features

For every incoming customer message, the system performs the following actions:

1.  **Accepts Customer Message**: Takes a raw customer support message as input.
2.  **Classifies Category**: Determines the primary category or intent of the ticket (e.g., `billing_and_payments`, `technical_issue`, `product_inquiry`).
3.  **Detects Priority**: Assigns a priority level (`low`, `medium`, `high`, `urgent`) based on the urgency and business impact of the issue.
4.  **Assigns Department**: Routes the ticket to the most appropriate internal department (e.g., `billing_team`, `technical_support`, `customer_success`).
5.  **Generates Issue Summary**: Creates a concise 1-2 sentence internal summary for support agents.
6.  **Generates Customer Reply**: Drafts a professional and empathetic customer-facing reply.
7.  **Decides Human Escalation**: Determines whether the ticket requires human intervention based on predefined rules (e.g., complaints, urgent issues, low model confidence).

## Intent-Based Routing Workflow

The core of this system is an **intent-based routing workflow**. The LLM first classifies the ticket's intent (category), which then drives all subsequent decisions regarding priority, department assignment, reply tone, and escalation rules.

## Interchangeable LLM Providers

The system supports **three interchangeable LLM providers**:

*   **OpenAI** (e.g., `gpt-4.1-mini`)
*   **Groq** (e.g., `llama-3.3-70b-versatile`)
*   **Gemini** (e.g., `gemini-2.5-flash`)

You can switch between these providers with a single parameter change, without modifying the core logic.

## Conversation Memory

To handle multi-turn conversations effectively, the system maintains a **conversation memory**. This allows the model to retain the full context of a ticket's history when a customer sends follow-up messages, enabling more relevant and consistent responses.

## Setup and Usage

To use this system, you'll need to:

1.  **Install Dependencies**: Install the required Python packages (OpenAI, Groq, Google Generative AI, Gradio, Pydantic).
2.  **API Keys**: Configure API keys for your chosen LLM providers (OpenAI, Groq, Gemini) in your Colab Secrets (or environment variables).
3.  **Run the Notebook**: Execute the notebook cells sequentially to initialize clients, define the structured output schema, set up the system prompt, and configure conversation memory.
4.  **Interactive Testing**: Use the provided `main()` function for console-based interactive testing.
5.  **Gradio Web UI**: Launch the Gradio web interface (`demo.launch()`) for an easy-to-use chatbot-style UI, allowing non-technical users to interact with the system and test different providers and ticket scenarios.